In [1]:
# Load environment variables (API key) and keep notebook output clean
import os, warnings
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
load_dotenv()

print("Key loaded:", bool(os.getenv("OPENAI_API_KEY")))


Key loaded: True


In [2]:
# Define a sample input that looks like a real support ticket
ticket_text = """
Hi team - my dashboard is super slow since yesterday.
It takes 20-30 seconds to load. I am on Windows laptop.
This started after the update. Please fix ASAP.
Also: error code TIMEOUT_504 sometimes shows up.
Thanks, Pinal
"""
print(ticket_text)



Hi team - my dashboard is super slow since yesterday.
It takes 20-30 seconds to load. I am on Windows laptop.
This started after the update. Please fix ASAP.
Also: error code TIMEOUT_504 sometimes shows up.
Thanks, Pinal



In [3]:
# Create the model component (the "engine" that generates the response)
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5-mini", temperature=0)


In [4]:
# Create a prompt template (the "shape" of the request we send to the model)
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You convert support tickets into structured incident summaries."),
    ("human", """
Extract fields from this ticket and respond in JSON with these keys only:
severity (low|medium|high),
product_area,
issue_summary,
suspected_cause,
next_action,
tags (array of short strings)

Ticket:
{ticket}
""")
])


In [5]:
# Define an output parser to convert model text into a Python dictionary
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser()


In [6]:
# Create a simple callback to observe what gets sent and received
from langchain_core.callbacks import BaseCallbackHandler

class SimpleLogger(BaseCallbackHandler):
    def on_chat_model_start(self, serialized, messages, **kwargs):
        print("\n--- Callback: Model Start ---")
        print("Messages being sent:")
        for m in messages[0]:
            print(f"[{m.type.upper()}] {m.content[:300]}")

    def on_chat_model_end(self, result, **kwargs):
        print("\n--- Callback: Model End ---")
        print("Model responded.")


In [7]:
# Build and run the chain imperatively (explicit: prompt -> model)
messages = prompt.format_messages(ticket=ticket_text)
response = llm.invoke(messages, config={"callbacks": [SimpleLogger()]})

raw_text = response.content
print("\n--- Raw model output ---")
print(raw_text)



--- Callback: Model Start ---
Messages being sent:
[SYSTEM] You convert support tickets into structured incident summaries.
[HUMAN] 
Extract fields from this ticket and respond in JSON with these keys only:
severity (low|medium|high),
product_area,
issue_summary,
suspected_cause,
next_action,
tags (array of short strings)

Ticket:

Hi team - my dashboard is super slow since yesterday.
It takes 20-30 seconds to load. I am on Wind

--- Raw model output ---
{
  "severity": "high",
  "product_area": "dashboard",
  "issue_summary": "Dashboard loading very slow (20-30s) since recent update; intermittent TIMEOUT_504 errors observed.",
  "suspected_cause": "Recent update introduced a regression causing backend/API performance degradation or gateway timeouts (504); potential server-side slowdown or increased request latency.",
  "next_action": "Reproduce issue on Windows, collect client logs and network traces, review backend/API latency and 504 error rates, correlate with recent deploys, esca

In [8]:
# Parse the raw model output into a structured Python dictionary
structured = parser.parse(raw_text)
print("--- Parsed result ---")
print(type(structured))
print(structured)

--- Parsed result ---
<class 'dict'>
{'severity': 'high', 'product_area': 'dashboard', 'issue_summary': 'Dashboard loading very slow (20-30s) since recent update; intermittent TIMEOUT_504 errors observed.', 'suspected_cause': 'Recent update introduced a regression causing backend/API performance degradation or gateway timeouts (504); potential server-side slowdown or increased request latency.', 'next_action': 'Reproduce issue on Windows, collect client logs and network traces, review backend/API latency and 504 error rates, correlate with recent deploys, escalate to backend team and prepare rollback or hotfix if confirmed.', 'tags': ['dashboard', 'performance', 'slow-loading', 'timeout-504', 'regression', 'windows', 'high-priority']}
